# 501 + 504 통합 FAISS DB 만들기

**목적**: 기존 `faiss_stat501_db`에 `stat504_full_data.txt`를 같은 방식으로 청킹하여 `add_texts()`로 직접 추가한 뒤,
`faiss_stat_integrated_db`로 저장한다.

**핵심**: 504만 따로 DB로 저장하지 않는다 — 501 DB에 바로 추가하는 게 가장 깔끔하다.

**입력**:
- 기존 501 DB: `C:/Users/xaxa0/opensource_statchatbot/vector_db/faiss_stat501_db/`
- 504 텍스트:  `C:/Users/xaxa0/opensource_statchatbot/vectordb_crawling/stat504_full_data.txt`

**출력**:
- 통합 DB: `C:/Users/xaxa0/opensource_statchatbot/vector_db/faiss_stat_integrated_db/`

**비용**: 504 본문(약 970KB)을 한 번 임베딩 — `text-embedding-3-small` 기준 대략 수 센트 수준.

In [ ]:
# 1) 라이브러리 임포트
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
import os

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

# 프로젝트 루트는 .env 파일이 있는 폴더 (어디서 실행해도 안전)
ENV_PATH = find_dotenv()
PROJECT_ROOT = Path(ENV_PATH).parent
load_dotenv(ENV_PATH)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, ".env 파일에 OPENAI_API_KEY가 없습니다."

# 3) 경로 세팅
txt_path_504        = str(PROJECT_ROOT / "data" / "stat504_full_data.txt")
db_path_501         = str(PROJECT_ROOT / "vector_db" / "faiss_stat501_db")
db_path_integrated  = str(PROJECT_ROOT / "vector_db" / "faiss_stat_integrated_db")

assert os.path.exists(txt_path_504), f"❌ 504 텍스트 파일 없음: {txt_path_504}"
assert os.path.exists(db_path_501), f"❌ 기존 501 DB 폴더 없음: {db_path_501}"
print("✅ 입력 경로 확인 완료")

In [ ]:
# 4) 504 텍스트 로드 + 청킹 (501과 동일 설정: chunk_size=1500, overlap=200)
with open(txt_path_504, "r", encoding="utf-8") as f:
    full_text_504 = f.read()

print(f"📂 504 텍스트 로드 완료 — 총 {len(full_text_504):,}자")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)
chunks_504 = text_splitter.split_text(full_text_504)
print(f"✂️  504 청크 개수: {len(chunks_504)}")

In [ ]:
# 5) 기존 501 FAISS DB 로드
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

main_db = FAISS.load_local(
    db_path_501,
    embeddings,
    allow_dangerous_deserialization=True,
)
print(f"📦 501 DB 로드 완료 — 기존 벡터 개수: {main_db.index.ntotal}")

In [ ]:
# 6) 504 텍스트를 기존 DB에 바로 추가 (별도 db_504 만들 필요 없음!)
print("🧠 504 청크 임베딩 + DB에 추가 중... (1~2분 소요)")
main_db.add_texts(chunks_504)
print(f"✅ 추가 완료 — 통합 후 벡터 개수: {main_db.index.ntotal}")

In [ ]:
# 7) 통합 DB 저장
main_db.save_local(db_path_integrated)
print(f"💾 통합 FAISS DB 저장 완료 → {db_path_integrated}")

In [ ]:
# 8) 검증 — 504 키워드(이산자료, 로지스틱 등)로 검색이 잘 되는지 확인
test_db = FAISS.load_local(db_path_integrated, embeddings, allow_dangerous_deserialization=True)

test_queries = [
    "로지스틱 회귀에서 오즈비(odds ratio)는 어떻게 해석하나요?",   # 504
    "다중공선성을 확인하는 VIF 지표는 무엇인가요?",                  # 501
    "분할표(contingency table)에서 카이제곱 검정은 어떻게 하나요?", # 504
]

for q in test_queries:
    print(f"\n🔍 질문: {q}")
    results = test_db.similarity_search(q, k=2)
    for i, doc in enumerate(results, 1):
        preview = doc.page_content[:200].replace("\n", " ")
        print(f"  [{i}] {preview}...")